<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/courses/c34-header.png" alt="Nextia Learning · Anomaly Detection: Find What Is Unusual" width="100%">

# Control charts: is this variation normal?

**[Control charts: is this variation normal?](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/)** · Anomaly Detection: Find What Is Unusual · Module 3, lesson 1 of 7 · [learning.nextia-ai.com](https://learning.nextia-ai.com)

**You will learn to** separate normal day-to-day variation from a real change with a control chart: limits from a stable reference period, then monitoring. You build the individuals chart (I-MR) of Larkfield's daily refund rate by hand, with pandas and NumPy.

| | |
|---|---|
| **Time** | About 40 minutes |
| **Needs** | An internet connection for the setup cells (less than 1 MB). No account. |
| **You should know** | Mean and standard deviation, from [Z-scores, and why the mean is fooled](https://learning.nextia-ai.com/courses/anomaly-detection/m02/z-scores-and-why-the-mean-is-fooled/). |
| **You do not need** | A local project, or a statistics package: every number is a few lines of pandas. |
| **Tested** | Python 3.12.13 in Jupyter on macOS (Apple silicon), from a fresh start (*Restart and run all*), with the package versions of the setup cell |

### How to use this notebook

This notebook has every query, task and check of the lesson. The [lesson page](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/) has the full explanations, the diagrams and the knowledge checks that count toward your certificate. Keep both open.

1. Run the cells in order, from the top. Press **Shift+Enter** to run a cell and move to the next one.
2. At a **Predict** note, write your prediction down before you run the cell.
3. At a **Your turn** note, write your answer in the cell, run it, then run the check cell below it.
4. If you are stuck, open the [solution notebook](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/notebooks/M03-L01-control-charts/control-charts-solution.ipynb).

## Setup: install the packages

Run the next cell. It installs the versions that the course uses: pandas 3.1.1, numpy 2.5.4, scipy 1.18.1, scikit-learn 1.9.1, matplotlib 3.11.2. It needs an internet connection. The first time, it takes up to a minute.

You should see one line per package, with the same versions. If you see other versions, restart the kernel (in Colab: **Runtime › Restart session**; in Kaggle: **Run › Restart & clear cell outputs**), then run this cell again. A restart is needed when an older version was already loaded.

In [ ]:
%pip install -q --disable-pip-version-check pandas==3.1.1 numpy==2.5.4 scipy==1.18.1 scikit-learn==1.9.1 matplotlib==3.11.2
from importlib.metadata import version
for p in ['pandas', 'numpy', 'scipy', 'scikit-learn', 'matplotlib']:
    print(p, version(p))

## Setup: get the course data

Run the next cell. It downloads the data that this lesson uses (`refunds_daily.csv`, `sealed-key.txt`) into a folder `anomaly-course/data`, and checks each file's checksum. It also downloads `compare.py`, the course's shared code. It needs an internet connection and no account. It takes a few seconds.

You should see `Ready:` and the files with their sizes. If you run the cell again, it uses the files that are already there.

If the cell shows an error, read its last line:

- `Could not download`: check your internet connection, then run the cell again. In Kaggle, turn on **Internet** in the notebook settings.
- `wrong checksum`: delete the folder `anomaly-course` (in Colab: the **Files** panel on the left), then run the cell again.

The data is made up for this course: no real people, cards or payments. [The dataset card](https://github.com/Nextia-Intelligence/nextia-learning-labs/blob/main/C34/data/dataset-card.md) describes every table.

In [ ]:
# Setup: download the course data into anomaly-course/data/ and check each file.
# Standard library only.
import hashlib, os, shutil, subprocess, urllib.request
from pathlib import Path

LABS = "https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/C34/"
FILES = {
    "refunds_daily.csv": "04c6384d1fbe8da908a739edc084eabc413581d75fce7cd84dcf7a90988996a6",
    "sealed-key.txt": "144cfd16737bdfcc4e992ad8ede0b8856265a2193c89fa7266fe0b27cb62ff98",
}
CODE_FILES = ["compare.py"]
HERE = Path.cwd()
# The notebook works in its own folder anomaly-course/, marked with a file, so it never writes into another project.
if Path(".nextia-notebook").exists():          # the cell ran before: start again from the parent folder
    os.chdir("..")
    HERE = Path.cwd()
WORK = Path("anomaly-course").resolve()
(WORK / "data").mkdir(parents=True, exist_ok=True)
(WORK / ".nextia-notebook").touch()

def sha256(path):
    return hashlib.sha256(path.read_bytes()).hexdigest()

def download(url, path):
    try:
        urllib.request.urlretrieve(url, path)
    except OSError:
        # Python without certificates (python.org on macOS): try curl.
        if subprocess.run(["curl", "-fsSL", "-o", str(path), url]).returncode:
            raise SystemExit(f"Could not download {url}. Check your internet connection.")

def fetch(name, url, path, expected=None):
    """Use a copy next to the notebook if there is one, else download it."""
    if path.exists() and (expected is None or sha256(path) == expected):
        return
    if (HERE / name).exists():
        shutil.copy(HERE / name, path)
    else:
        download(url, path)
    if expected and sha256(path) != expected:
        raise SystemExit(f"{name} has the wrong checksum. Delete the folder anomaly-course and run again.")

for name, expected in FILES.items():
    fetch(name, LABS + "data/" + name, WORK / "data" / name, expected)
for name in CODE_FILES:
    fetch(name, LABS + name, WORK / name)
os.chdir(WORK)
print("Ready:", ", ".join(f"{n} ({(Path('data') / n).stat().st_size / 1e6:.1f} MB)" for n in FILES))

## Setup: helpers

The next cell defines the check helpers. They tell you if your answer is right, without showing the answer:

- `expect(what, yours, expected)` compares one of your numbers with the expected one, and says "too high" or "too low".
- `expect_hash(what, yours, expected)` compares any result with a hidden fingerprint of the expected one.

It also loads pandas as `pd`, NumPy as `np` and matplotlib as `plt`. Run the cell. It prints nothing.

In [ ]:
# Helpers: check an answer without showing it, and the chart style.
import hashlib, json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
pd.set_option("display.precision", 3)
pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 20)
warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams.update({"figure.figsize": (9, 3.2), "axes.spines.top": False, "axes.spines.right": False})

def fingerprint(value):
    def norm(v):
        if hasattr(v, "to_dict") and hasattr(v, "columns"):
            return {"columns": [str(c) for c in v.columns], "rows": [norm(list(r)) for r in v.itertuples(index=False)]}
        if hasattr(v, "tolist"):
            return norm(v.tolist())
        if isinstance(v, dict):
            return {str(k): norm(x) for k, x in sorted(v.items(), key=lambda kv: str(kv[0]))}
        if isinstance(v, (list, tuple, set)):
            return [norm(x) for x in (sorted(v, key=str) if isinstance(v, set) else v)]
        if isinstance(v, bool) or v is None or isinstance(v, int):
            return v
        if isinstance(v, float):
            return None if v != v else round(v, 4)
        if hasattr(v, "item"):
            return norm(v.item())
        return str(v)
    return hashlib.sha256(json.dumps(norm(value), sort_keys=True).encode()).hexdigest()[:16]

def expect(what, yours, expected, tolerance=1e-6):
    """Compare one number or value with the expected one."""
    if isinstance(yours, (np.integer, np.floating)):
        yours = yours.item()
    if isinstance(expected, float) and isinstance(yours, (int, float)):
        same = abs(yours - expected) <= tolerance * max(1, abs(expected))
    else:
        same = yours == expected
    if same:
        print(f"Check passed: {what} is {yours}.")
    elif isinstance(expected, (int, float)) and isinstance(yours, (int, float)):
        print(f"Not yet: {what} is {yours}, which is too {'high' if yours > expected else 'low'}.")
    else:
        print(f"Not yet: {what} is {yours!r}, which is not the expected value.")

def expect_hash(what, yours, expected):
    """Compare any result with the fingerprint of the expected one."""
    if yours is None or yours is Ellipsis:
        print(f"Not yet: {what} has no value. Write your answer in the cell above, then run it again.")
    elif fingerprint(yours) == expected:
        print(f"Check passed: {what} is right.")
    else:
        print(f"Not yet: {what} is not the expected result. Read the task again, and check each step.")

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

## 1. 2.1% last week, 2.4% this week: a problem?

Grace, the support lead, looks at Larkfield's refund rate every Monday: the refunds of the week divided by its orders. Run the next cell. It prints the weekly rate (in %) for seven weeks of autumn 2024, when nothing changed at Larkfield.

> **Check.** You should see rates from 1.65 to 2.24.

In [ ]:
import compare
daily = pd.read_csv("data/refunds_daily.csv", parse_dates=["date"]).set_index("date")
weekly = daily.dropna().resample("W-SUN")[["orders", "refunds"]].sum()
weekly["rate"] = weekly.refunds / weekly.orders
print((weekly.rate.loc["2024-10-01":"2024-11-17"] * 100).round(2).to_string())

Nothing changed, and the rate still moved between 1.65% and 2.24%. This is **common-cause variation**: the normal noise of a process (some weeks have a few more damaged parcels, by chance). A **special cause** is something new: a new carrier, a broken feed, an agent who refunds too much. Reacting to common-cause noise wastes time; missing a special cause costs money.

A **control chart** draws the measure every day with a centre line and two **control limits**. Inside the limits is the usual noise. A point outside is a signal: look for a special cause. This is **statistical process control (SPC)**: factories, call centres and banks have used it for a hundred years.

## 2. The idea on ten days

The **individuals and moving range chart (I-MR)** needs only the values. The centre line is their mean. The spread comes from the **moving range**: the size of the change from one day to the next. Its mean, divided by 1.128 (a constant for ranges of two values), estimates the standard deviation of the noise. The limits are 3 of these sigmas from the centre.

Run the next cell. It computes the chart for ten days of refund rates.

> **Check.** You should see a centre line of 2.10, a mean moving range of 0.400 and limits from 1.04 to 3.16.

In [ ]:
rates = np.array([2.0, 2.4, 1.8, 2.2, 2.1, 1.9, 2.5, 2.0, 2.3, 1.8])   # ten days, refund rate in %
center = rates.mean()
moving_range = np.abs(np.diff(rates))                 # the change from one day to the next
sigma = moving_range.mean() / 1.128                  # 1.128 turns the mean moving range into a standard deviation
print("centre line:", round(center, 2), "| mean moving range:", round(moving_range.mean(), 3), "| sigma:", round(sigma, 3))
print("control limits:", round(center - 3 * sigma, 2), "to", round(center + 3 * sigma, 2))

Why the moving range, and not the usual standard deviation of all the days? Because a real shift inside the data would make the standard deviation large, and the limits too wide to see it. The changes from day to day stay small, even when the level moves.

## 3. Phase I: limits from a stable period

The limits come from a **reference period** when the process was stable (phase I). Here: April to December 2024. After that, the chart **monitors** new days against these fixed limits (phase II).

> **Check.** You should see 275 days, a centre of 0.0214 and limits 0.0028 to 0.0400.

In [ ]:
phase1 = daily.loc["2024-04-01":"2024-12-31", "refund_rate"].dropna()      # the stable reference period
center = phase1.mean()
sigma = phase1.diff().abs().mean() / 1.128
lcl, ucl = center - 3 * sigma, center + 3 * sigma
print(f"phase I: {len(phase1)} days; centre {center:.4f}, sigma {sigma:.4f}, limits {lcl:.4f} to {ucl:.4f}")

The rate of a day is between 0.28% and 4.00% when nothing special happens. Control limits are **not targets** ("we want 2%") and **not specification limits** ("never above 3%"): they describe what the process does, not what we wish.

## 4. The chart, phase I and phase II

Run the next cell. It draws every day from April 2024, with the limits from phase I, and marks the days outside them.

In [ ]:
rate = daily.refund_rate.loc["2024-04-01":].dropna()
outside = (rate > ucl) | (rate < lcl)
fig, ax = plt.subplots(figsize=(10, 3.4))
ax.plot(rate.index, rate * 100, lw=0.6, color="grey")
ax.scatter(rate.index[outside], rate[outside] * 100, color="red", zorder=3, s=14, label="outside the limits")
for y in (lcl, center, ucl):
    ax.axhline(y * 100, color="black", lw=0.8, ls="--" if y != center else "-")
ax.axvline(pd.Timestamp("2025-01-01"), color="blue", lw=0.8)
ax.text(pd.Timestamp("2025-01-05"), ucl * 100 + 0.2, "phase II: monitoring", color="blue", fontsize=8)
ax.set_ylabel("refund rate (%)")
ax.legend(loc="upper left")
plt.show()
print("days outside the limits:", [str(d.date()) for d in rate.index[outside]])

> **Predict.** We planted a lasting rise of 0.3 points in the refund rate from 2025-04-07 (a new delivery carrier damages more parcels). How many days after the shift does this chart first go above its upper limit? Write a number, then run the next cell.

In [ ]:
phase2 = rate.loc["2025-01-01":]
high = phase2 > ucl
report = compare.alarm_report(phase2.index, high, "2025-04-07", ("2025-01-01", "2025-04-06"))
print(report)
before, after = phase2.loc[:"2025-04-06"].mean(), phase2.loc["2025-04-07":].mean()
print(f"mean rate before the shift {before:.4f}, after {after:.4f}")

> **Check.** You should see `'first alarm': '2025-07-14'` and `'days to alarm': 98`.

98 days. The mean rate rose from 0.0211 to 0.0244, but one day's noise (sigma 0.0062) is larger than the shift (0.003), so a single day rarely crosses the limit. A Shewhart chart (one point against 3 sigma) is good at **big, sudden** changes and slow at **small, lasting** ones. Lesson 4 of this module builds the CUSUM and EWMA charts, which add up small differences and catch this shift much sooner.

## 5. Change one thing: 2-sigma limits

Narrower limits catch more, but they alarm more often when nothing happened. Run the next cell. It counts the alarms in the stable months of phase II (January to early April 2025) with 3-sigma and with 2-sigma limits.

In [ ]:
stable = rate.loc["2025-01-01":"2025-04-06"]
for k in (3, 2):
    alarms = int(((stable - center).abs() > k * sigma).sum())
    print(f"limits at {k} sigma: {alarms} alarms in {len(stable)} stable days ({alarms / len(stable) * 30.44:.1f} a month)")

> **Check.** You should see 7 alarms with 2-sigma limits.

Every alarm in these months is a **false alarm**: nothing changed. The price of a chart is its false alarms per month; lesson 2 measures it properly with the average run length.

## 6. A common mistake: limits from every day

If the reference period contains the problems you want to find, they widen the limits. Run the next cell.

In [ ]:
all_days = daily.refund_rate.dropna()               # the WRONG reference: every day, the shift and the broken feed too
center_all = all_days.mean()
sigma_all = all_days.diff().abs().mean() / 1.128
print(f"limits from every day: {center_all - 3 * sigma_all:.4f} to {center_all + 3 * sigma_all:.4f}")
print(f"limits from phase I:   {lcl:.4f} to {ucl:.4f}")

The limits from every day are wider: the shift and the broken feed (a day with a rate of zero) are inside the "normal" now. Compute limits from a period you have checked, and keep them fixed while you monitor.

## 7. Check yourself

**Your turn 1.** What is the centre line of phase I? Write it as `center_line`, rounded to 4 decimals.

In [ ]:
center_line = ...   # your answer

In [ ]:
expect('center_line', center_line, 0.0214, tolerance=0.0001)

**Your turn 2.** How many days of the stable months (2025-01-01 to 2025-04-06) are outside the 3-sigma limits, on either side? Write the number as `stable_alarms`.

In [ ]:
stable_alarms = ...   # your code

In [ ]:
expect('stable_alarms', stable_alarms, 0)

**Your turn 3.** List the days of phase II (from 2025-01-01) **below** the lower limit, as strings `"YYYY-MM-DD"`, in a list `low_days`. Then look one up in the data: what special cause could make the refund rate fall to that level?

In [ ]:
low_days = ...   # your code

In [ ]:
expect_hash('low_days', low_days, '4b01fd7782b22d60')

A rate of 0 on 2025-11-10 is not good news: on that day the refunds feed sent zeros (an anomaly we planted). A low point is a signal too.

## Recap

- Common-cause variation is the usual noise; a special cause is something new. A control chart separates the two with limits from a stable period.
- The I-MR chart uses the moving range, so a shift inside the data does not hide itself.
- One point against 3-sigma limits found the planted 0.3-point rise only after 98 days. Small, lasting shifts need other charts.

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/section-divider-light.png" alt="" width="100%">

**Next lesson:** [Rules for patterns, and the price of each rule](https://learning.nextia-ai.com/courses/anomaly-detection/m03/rules-for-patterns/) · **This lesson:** [read it on the site](https://learning.nextia-ai.com/courses/anomaly-detection/m03/control-charts/)

<img src="https://raw.githubusercontent.com/Nextia-Intelligence/nextia-learning-labs/main/nextia-notebook-branding/assets/notebook-footer-light.png" alt="Nextia Learning · learning.nextia-ai.com" width="100%">

Free, practical AI courses at [learning.nextia-ai.com](https://learning.nextia-ai.com). The practice data is synthetic and has no real people.